In [1]:
import os
import warnings

import albumentations as A
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



Device: cuda


In [2]:
# Paths (kept compatible with Kaggle dataset layout)
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
train_csv_path = "../input/cassava-leaf-disease-classification/train.csv"
train_images_path = "../input/cassava-leaf-disease-classification/train_images"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

assert os.path.exists(sample_sub_path), f"Missing: {sample_sub_path}"
assert os.path.exists(train_csv_path), f"Missing: {train_csv_path}"
assert os.path.isdir(train_images_path), f"Missing dir: {train_images_path}"
assert os.path.isdir(test_images_path), f"Missing dir: {test_images_path}"

# We keep checkpoint discovery, but we no longer depend on it to achieve a non-random model.
requested_model_path = "../input/en-b4-tta-calr-clahe/model(14).pth"


def find_checkpoint(preferred_path: str):
    if preferred_path and os.path.exists(preferred_path):
        return preferred_path

    search_roots = ["../input", "/kaggle/input"]
    exts = (".pth", ".pt", ".bin")
    candidates = []
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for fn in filenames:
                if fn.lower().endswith(exts):
                    candidates.append(os.path.join(dirpath, fn))
    if not candidates:
        return None

    def score(p):
        s = p.lower()
        sc = 0
        if "cassava" in s or "leaf" in s or "disease" in s:
            sc += 6
        if "efficientnet" in s:
            sc += 3
        if "b4" in s:
            sc += 3
        if "fold" in s or "epoch" in s or "best" in s or "model" in s:
            sc += 1
        if "imagenet" in s:
            sc -= 2
        return (-sc, len(p))

    candidates = sorted(candidates, key=score)
    return candidates[0]


model_path = find_checkpoint(requested_model_path)
print("Requested checkpoint:", requested_model_path)
print("Resolved checkpoint:", model_path)



Requested checkpoint: ../input/en-b4-tta-calr-clahe/model(14).pth
Resolved checkpoint: None


In [3]:
# Minimal, score-relevant change:
# - Always start from ImageNet-pretrained EfficientNet-B4 (same architecture core),
# - then train/fine-tune the existing 5-class head (and lightly fine-tune) on the provided train set.
# This fixes the main issue (random 5-class head) without changing the approach (EffNetB4 + 5-class head).
model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)
model.to(DEVICE)

# Optional: if a compatible checkpoint exists, load it (keeps previous logic).
# If it doesn't exist or is incompatible, training below will still produce a non-random model.
if model_path is not None:
    try:
        ckpt = torch.load(model_path, map_location="cpu")
        if isinstance(ckpt, dict) and ("state_dict" in ckpt or "model" in ckpt):
            state = ckpt.get("state_dict", ckpt.get("model"))
        elif isinstance(ckpt, dict):
            state = ckpt
        else:
            raise ValueError("Unsupported checkpoint format")

        new_state = {}
        for k, v in state.items():
            nk = k
            if nk.startswith("module."):
                nk = nk[len("module.") :]
            new_state[nk] = v

        head_w_key = "classifier.1.weight"
        head_b_key = "classifier.1.bias"
        if head_w_key in new_state and isinstance(new_state[head_w_key], torch.Tensor):
            if new_state[head_w_key].shape[0] != 5:
                print(
                    f"Checkpoint head has {new_state[head_w_key].shape[0]} classes; expected 5. "
                    "Ignoring head weights and using ImageNet init + training."
                )
                new_state.pop(head_w_key, None)
                new_state.pop(head_b_key, None)

        missing, unexpected = model.load_state_dict(new_state, strict=False)
        print("Loaded checkpoint (partial allowed).")
        print("Missing keys:", len(missing), "Unexpected keys:", len(unexpected))
    except Exception as e:
        print("Checkpoint load failed; will rely on training. Error:", repr(e))



Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 111MB/s]


In [4]:
# EfficientNet-B4 expects 380x380; feeding arbitrary sizes can hurt performance.
IMG_SIZE = 380

train_norm = A.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
    max_pixel_value=255.0,
    p=1.0,
)

# Keep inference deterministic TTA (same as your current intent), but add resize to match the model.
tta_transforms = [
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), train_norm]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1.0), train_norm]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1.0), train_norm]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Transpose(p=1.0), train_norm]),
    A.Compose(
        [
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Transpose(p=1.0),
            A.HorizontalFlip(p=1.0),
            train_norm,
        ]
    ),
]
tta_count = len(tta_transforms)

# For training, use light augmentation (core logic unchanged: still supervised CE on labels).
train_tfm = A.Compose(
    [
        A.RandomResizedCrop(
            size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0
        ),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.1),
        train_norm,
    ]
)

val_tfm = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), train_norm])


def albumentations_image_to_tensor(img_hwc: np.ndarray) -> torch.Tensor:
    if img_hwc.ndim != 3 or img_hwc.shape[2] != 3:
        raise ValueError(f"Unexpected image shape: {img_hwc.shape}")
    x = torch.from_numpy(img_hwc).permute(2, 0, 1).contiguous()  # CHW
    return x.float()




In [5]:
from torch.utils.data import Dataset, DataLoader


class CassavaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, images_dir: str, tfm):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.tfm = tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.images_dir, row.image_id)
        img = Image.open(img_path).convert("RGB")
        img = np.array(img)
        img = self.tfm(image=img)["image"]
        x = albumentations_image_to_tensor(img)
        y = int(row.label)
        return x, y


train_df = pd.read_csv(train_csv_path)

# Minimal, deterministic split to enable training without changing overall semantics.
# (We do not add early stopping; fixed epochs to keep behavior stable.)
perm = np.random.RandomState(42).permutation(len(train_df))
split = int(0.9 * len(train_df))
tr_idx, va_idx = perm[:split], perm[split:]
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
va_df = train_df.iloc[va_idx].reset_index(drop=True)

train_ds = CassavaDataset(tr_df, train_images_path, train_tfm)
val_ds = CassavaDataset(va_df, train_images_path, val_tfm)

BATCH_SIZE = 16 if torch.cuda.is_available() else 8
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)



In [6]:
# Minimal training addition to move score toward target:
# - CrossEntropyLoss + AdamW
# - A small, fixed number of epochs (no early stopping), to keep runtime < 600s and avoid over-optimization.
criterion = nn.CrossEntropyLoss()

# Light fine-tune: smaller LR for backbone, larger LR for head; same model and loss.
head_params = list(model.classifier[1].parameters())
backbone_params = [
    p for n, p in model.named_parameters() if not n.startswith("classifier.1.")
]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 3e-5},
        {"params": head_params, "lr": 3e-4},
    ],
    weight_decay=1e-4,
)


def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()


EPOCHS = 2  # small fixed number: enough to escape random head and move toward target without chasing best possible

for epoch in range(EPOCHS):
    model.train()
    tr_loss = 0.0
    tr_acc = 0.0
    n_tr = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        tr_loss += loss.item() * bs
        tr_acc += accuracy_from_logits(logits.detach(), yb) * bs
        n_tr += bs

    model.eval()
    va_loss = 0.0
    va_acc = 0.0
    n_va = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            bs = xb.size(0)
            va_loss += loss.item() * bs
            va_acc += accuracy_from_logits(logits, yb) * bs
            n_va += bs

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"train_loss {tr_loss/n_tr:.4f} train_acc {tr_acc/n_tr:.4f} | "
        f"val_loss {va_loss/n_va:.4f} val_acc {va_acc/n_va:.4f}"
    )

model.eval()



Epoch 1/2 | train_loss 0.7959 train_acc 0.7111 | val_loss 0.5505 val_acc 0.8019
Epoch 2/2 | train_loss 0.5348 train_acc 0.8102 | val_loss 0.4629 val_acc 0.8409


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [7]:
sample_sub = pd.read_csv(sample_sub_path)

predictions = []

with torch.no_grad():
    for _, sample_row in sample_sub.iterrows():
        img_path = os.path.join(test_images_path, sample_row.image_id)
        base_img = Image.open(img_path).convert("RGB")
        base_img = np.array(base_img)

        image_logits_sum = None
        for tfm in tta_transforms:
            aug_img = tfm(image=base_img)["image"]  # float32, normalized HWC
            x = albumentations_image_to_tensor(aug_img).to(DEVICE)  # (3,H,W)
            outputs = model(x.unsqueeze(0))  # (1,5)

            if image_logits_sum is None:
                image_logits_sum = outputs
            else:
                image_logits_sum = image_logits_sum + outputs

        image_pred = image_logits_sum / float(tta_count)
        pred_label = int(torch.argmax(image_pred, dim=1).item())
        predictions.append([sample_row.image_id, pred_label])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
sub_df.to_csv("submission.csv", index=False)
print(sub_df.head())
print("Wrote submission.csv with shape:", sub_df.shape)
print(
    "TTA count:",
    tta_count,
    "| Using checkpoint:",
    (model_path is not None),
    "| Epochs trained:",
    EPOCHS,
)

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      4
4  1234571117.jpg      3
Wrote submission.csv with shape: (2676, 2)
TTA count: 5 | Using checkpoint: False | Epochs trained: 2
